# 03. Difficulty Scaling

The archived `analysis_A_difficulty_results.csv` encodes one EXP-15 difficulty analysis, but the audited manuscript pass recomputed the paper-facing figure directly from `unified_metrics.csv` because the archive bins and comparison pairs drifted. This notebook shows both.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

analysis_a = pd.read_csv(DATA / 'analysis' / 'analysis_A_difficulty_results.csv')
unified = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
analysis_a.head()
        


In [ ]:
truth_mag = unified[['problem_id', 'truth']].drop_duplicates().assign(abs_truth=lambda x: x['truth'].abs())
quartiles = truth_mag['abs_truth'].quantile([0.25, 0.50, 0.75]).to_list()

def assign_bin(value):
    if value <= quartiles[0]:
        return 'Small'
    if value <= quartiles[1]:
        return 'Medium'
    if value <= quartiles[2]:
        return 'Large'
    return 'Extra Large'

truth_mag['difficulty_bin'] = truth_mag['abs_truth'].apply(assign_bin)
merged = unified.merge(truth_mag[['problem_id', 'difficulty_bin']], on='problem_id', how='left')
rows = []
for difficulty_bin, bin_df in merged.groupby('difficulty_bin'):
    peak = None
    for layer, layer_df in bin_df.groupby('layer'):
        g4 = layer_df[layer_df['group'] == 'G4']['radius_of_gyration']
        g1 = layer_df[layer_df['group'] == 'G1']['radius_of_gyration']
        d = cohens_d(g4, g1)
        if peak is None or abs(d) > abs(peak['cohens_d']):
            peak = {'difficulty_bin': difficulty_bin, 'layer': int(layer), 'cohens_d': float(d)}
    rows.append(peak)
peak_df = pd.DataFrame(rows).sort_values('difficulty_bin', key=lambda s: s.map({'Small':0,'Medium':1,'Large':2,'Extra Large':3}))
peak_df
        


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=peak_df, x='difficulty_bin', y='cohens_d', palette='Blues_d')
for idx, row in peak_df.reset_index(drop=True).iterrows():
    ax.text(idx, row['cohens_d'] + 0.15, f"d={row['cohens_d']:.2f}
L{row['layer']}", ha='center', va='bottom', fontsize=10)
plt.axhline(0.8, color='black', linestyle='--', linewidth=1)
plt.text(3.1, 0.88, "Large effect (Cohen)", ha='right', fontsize=9)
plt.title('Peak radius-of-gyration separation by quartile-defined difficulty bin')
plt.xlabel('Difficulty bin')
plt.ylabel("Peak Cohen's d (G4 vs G1)")
plt.tight_layout()
plt.savefig(FIGURES / 'repro_03_difficulty_scaling.png', dpi=300)
plt.show()

print('Archive analysis_A preview (often G4 vs G2 in the local archive):')
display(analysis_a[analysis_a['metric'] == 'radius_of_gyration'].sort_values(['mag_bin', 'layer']).head(12))
        
